# BCI Toolbox — Tutorial

**Bayesian causal inference for multisensory research, from Python.**

This notebook walks through everything the programmatic API can do, in the order
you would actually use it:

| Part | Topic |
| --- | --- |
| 1 | The model in one picture |
| 2 | Loading data in any layout |
| 3 | Building a model step by step |
| 4 | Simulation |
| 5 | Fitting |
| 6 | Comparing decision strategies |
| 7 | Comparing models |
| 8 | When a modality is never reported (SIFI) |
| 9 | Many participants |
| 10 | Parameter recovery |
| 11 | Diagnostics: is my fit trustworthy? |
| 12 | Posterior predictive checks |
| 13 | Two stimulus dimensions, and how to see them |
| 14 | Judging the causal structure itself (unity, rubber hand) |
| 15 | More than two modalities |
| 16 | Continuous responses (spatial localisation) |
| 17 | Objectives and optimisers |
| 18 | Saving and reproducibility |

Everything runs on simulated or packaged data, so you can execute the whole
notebook without any files of your own.

> **Runtime.** The whole notebook takes a couple of minutes. Fits here use a
> small `n_sim` to stay quick; use `n_sim=10000` for results you intend to
> report.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import bcitoolbox as btb

REQUIRED = "0.3.0"
found = getattr(btb, "__version__", "older than 0.3.0")
print("BCI Toolbox", found, "|", btb.__file__)

if found != REQUIRED:
    raise RuntimeError(
        "This notebook needs BCI Toolbox " + REQUIRED + " but found " + found
        + ". If you just upgraded, restart the kernel (Kernel -> Restart):"
        + " Python caches imported modules, so a running kernel keeps serving"
        + " the old version. Otherwise run: pip install --upgrade bcitoolbox"
    )

btb.plot.set_style("notebook")

> **If that cell failed**, the notebook is newer than the toolbox in your
> kernel. Two causes, both quick to fix:
>
> 1. **A stale kernel.** Python caches imported modules, so a kernel started
>    before an upgrade keeps serving the old code - which shows up as
>    `AttributeError: module 'bcitoolbox.viz' has no attribute ...` on the newer
>    cells further down. Fix: **Kernel → Restart Kernel and Clear Outputs**,
>    then run from the top.
> 2. **An older copy shadowing the new one.** Fix: `pip install --upgrade
>    bcitoolbox`, then restart the kernel.

Not sure the installation is healthy? One command checks the numerical core
against the classic implementation and runs a parameter-recovery test:

```python
btb.selftest()
```

---
## 1. The model in one picture

An observer receives two noisy signals. They may come from **one** source or
from **two**. The observer infers which, and reports accordingly.

Three things drive everything else:

* `p_common` — how likely a common cause is *a priori*;
* `sigma_<modality>` — how noisy each sense is;
* the **disparity** between the two signals on this trial.

Let us sweep the disparity and watch the posterior probability of a common
cause collapse.

In [ ]:
model = btb.Model(["visual", "auditory"], ["space"], response="continuous")
model.fix("sigma_visual", 2.0)      # vision is precise
model.fix("sigma_auditory", 9.0)    # hearing is not
model.set_prior(mu=0.0, sigma=15.0)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for p in (0.2, 0.5, 0.8):
    btb.plot.posterior_common(model, values={"p_common": p},
                              disparities=np.linspace(-40, 40, 41),
                              label="p_common = %.1f" % p, ax=axes[0])
axes[0].set_title("inferring a common cause")

sim = model.simulate([(-10.0, 10.0)], values={"p_common": 0.5}, n=8000)
btb.plot.simulation(sim, ax=axes[1])
axes[1].set_title("responses at 20 deg disparity")
fig.tight_layout()

Read the right panel carefully: the **visual** response (blue) stays near its
own stimulus at −10, while the **auditory** response (red) is dragged towards
it. The unreliable modality gives way — that is the ventriloquism effect, and
here it falls out of the model rather than being built in.

---
## 2. Loading data in any layout

A `Data` object always has the same shape:

```
stimulus : (n_trials, n_modalities, n_dimensions)
response : (n_trials, n_modalities, n_dimensions)
```

with two conventions doing all the work:

* `NaN` in **stimulus** — that modality was not presented (a unimodal trial);
* `NaN` in **response** — that response was not collected.

The toolbox ships a small numerosity data set in the classic four-column
layout.

In [ ]:
data = btb.demo_data()
print(data.summary())

`read_data` reads any layout. The three most common:

```python
# 1. named columns (the usual case)
data = btb.read_data("sub01.csv",
                     stimulus={"visual": "loc_v", "auditory": "loc_a"},
                     response={"visual": "resp_v", "auditory": "resp_a"},
                     subject="participant")

# 2. the classic headerless four-column file
data = btb.read_data("demo.csv", layout="legacy",
                     modalities=["visual", "auditory"],
                     dimensions=["numerosity"])

# 3. two stimulus dimensions
data = btb.read_data("av2d.csv",
                     stimulus={"visual":   {"numerosity": "F", "time": "tF"},
                               "auditory": {"numerosity": "B", "time": "tB"}},
                     response={"visual":   {"numerosity": "rF"},
                               "auditory": {"numerosity": "rB"}},
                     missing=0, missing_dimension="numerosity")
```

Useful things a `Data` object knows about itself:

In [ ]:
print("conditions          :", data.n_conditions)
print("trials per condition:", data.condition_counts()[:5], "...")
print("response levels     :", data.response_levels())
print("response type       :", data.suggest_response_kind())
print("ever reported       :", data.reported_modalities)
print("presented but mute  :", data.unreported_modalities)
data.to_frame().head()

In [ ]:
btb.plot.data(data, max_conditions=9);

---
## 3. Building a model step by step

`Model.from_data` mirrors the architecture of a data set and scales the
starting values and bounds to it. Everything can then be adjusted by name — no
parameter ordering to remember.

In [ ]:
model = btb.Model.from_data(data, name="numerosity model")
print(model.summary())

The parameter names come from the architecture you declared:

| Name | Meaning |
| --- | --- |
| `p_common` | prior probability of a common cause |
| `sigma_<modality>` | sensory noise |
| `mu_prior`, `sigma_prior` | the prior over the source |
| `bias_<modality>` | constant sensory bias (fixed at 0) |
| `sigma_motor_<modality>` | response noise (fixed at 0) |
| `lapse` | random-response rate (fixed at 0) |
| `p_cutoff` | threshold for the `selection` strategy |

Any of them can be freed, fixed, bounded or tied:

In [ ]:
model.fix("p_cutoff", 0.5)                          # hold constant
model.free("p_common", bounds=(0.0, 1.0))           # estimate
model.set_param("sigma_visual", init=0.5, bounds=(0.05, 3.0))
model.set_prior(mu=2.0, sigma=3.0)                  # a number fixes it

print(model.params.to_frame().to_string(index=False))

`set_prior` accepts three shorthands, and so does anything else that takes a
parameter specification:

| You write | It means |
| --- | --- |
| `sigma=3.0` | fix at 3.0 |
| `sigma="free"` | estimate, keep current bounds |
| `sigma=(2.0, 0.5, 10.0)` | estimate, start at 2.0, bounds (0.5, 10.0) |

Two more useful moves — tying parameters together, and adding nuisance
components:

```python
model.tie("sigma_auditory", "sigma_visual")   # equal-noise model
model.add_lapse()                             # estimate a lapse rate
model.add_motor_noise()                       # estimate response noise
model.add_bias(modality="auditory")           # estimate a sensory bias
```

---
## 4. Simulation

Simulation needs no data at all — declare an architecture, set parameters,
choose conditions. This is how you develop intuition, design an experiment, or
generate predictions for a figure.

In [ ]:
generator = btb.Model(["visual", "auditory"], ["numerosity"])
generator.set_response("discrete", levels=[0, 1, 2, 3, 4])
for name, value in dict(p_common=0.65, sigma_visual=0.45, sigma_auditory=0.9,
                        mu_prior=2.0, sigma_prior=3.0).items():
    generator.fix(name, value)

conditions = [(v, a) for v in (1., 2., 3.) for a in (1., 2., 3.)]
conditions += [(v, np.nan) for v in (1., 2., 3.)]     # unimodal visual
conditions += [(np.nan, a) for a in (1., 2., 3.)]     # unimodal auditory

sim = generator.simulate(conditions, n=4000, seed=7)
sim.summary().head(6)

`NaN` in a condition means "this modality was not presented". Notice that the
unimodal rows below get no response for the absent modality, and their
`p_common` equals the prior — with a single signal there is nothing to infer.

In [ ]:
sim.summary().tail(6)

In [ ]:
btb.plot.simulation(sim, max_conditions=9);

A simulation converts straight into a data set, which is what makes parameter
recovery and power analysis easy (Part 10).

In [ ]:
simulated_data = sim.to_data("simulated numerosity")
print(simulated_data)
sim.to_frame().head()

---
## 5. Fitting

One call. The only argument you normally change is `n_sim`.

In [ ]:
model = btb.Model.from_data(data)
model.set_prior(mu=2.0, sigma=3.0)      # weakly identified here - see Part 11

fit = model.fit(data, n_sim=3000)
print(fit.summary())

The `Fit` object carries everything:

| Attribute | Meaning |
| --- | --- |
| `fit.values` | every parameter, fitted and fixed |
| `fit.error` | the objective at the optimum |
| `fit.log_likelihood`, `fit.aic`, `fit.bic` | information criteria |
| `fit.r2` | agreement between predicted and observed distributions |
| `fit.model` | a **copy** of the model with the fitted values |
| `fit.optimization` | every starting point, evaluation count, runtime |

In [ ]:
print("p_common =", round(fit["p_common"], 3))
print("free parameters:", dict((k, round(v, 3)) for k, v in fit.free_values.items()))
btb.plot.fit(fit, max_conditions=9);

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
btb.plot.params(fit, ax=axes[0])
btb.plot.bias(fit, ax=axes[1])
fig.tight_layout()

The right-hand panel is the ventriloquism curve, and it shows the signature of
causal inference: the bias of the noisier modality grows with disparity and
then **saturates** — a forced-fusion model would predict a straight line.

---
## 6. Comparing decision strategies

Having inferred *whether* there is a common cause, what does the observer do
with that belief?

| Strategy | Behaviour |
| --- | --- |
| `averaging` | weight the two estimates by their probabilities (statistically optimal) |
| `selection` | commit to the more probable causal structure |
| `matching` | pick one at random, in proportion to its probability |

Fit all three in one call and the best one is returned.

In [ ]:
comparison = model.fit(data, strategy="all", n_sim=2000)
print("winner:", comparison.strategy)
for entry in comparison.strategy_comparison:
    print("  {0:<10} error={1:9.2f}   BIC={2:9.1f}".format(
        entry["strategy"], entry["error"], entry["bic"]))

btb.plot.model_comparison(comparison);

---
## 7. Comparing models

Any two models fitted to the **same data** can be compared. A useful reference
is the model that never integrates: `p_common = 0`.

In [ ]:
null_model = btb.Model.from_data(data)
null_model.set_prior(mu=2.0, sigma=3.0)
null_model.fix("p_common", 0.0)
null_fit = null_model.fit(data, n_sim=2000)

btb.compare({"causal inference": fit, "no integration": null_fit})

`delta_bic` is the difference from the best model and `weight` is the
corresponding Schwarz weight — the posterior probability of each model, if you
had no prior preference between them.

> Only likelihood objectives (`objective="mll"`, the default) define AIC and
> BIC. Fits made with `"sse"`, `"r2"` or `"emd"` report `NaN` rather than a
> number that looks usable but is not.

---
## 8. When a modality is never reported

In the sound-induced flash illusion participants report only the number of
**flashes**. There is no auditory response column at all.

The toolbox fits anyway, and tells you exactly which parameters the data cannot
constrain.

In [ ]:
import warnings

responses = data.response.copy()
responses[:, 1, :] = np.nan                      # delete the auditory reports
sifi = btb.Data(data.stimulus, responses, data.modalities, data.dimensions,
                name="sifi", drop_absent_responses=False)

print("unreported modalities:", sifi.unreported_modalities)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sifi_model = btb.Model.from_data(sifi)
    sifi_model.set_prior(mu=2.0, sigma=3.0)
    sifi_fit = sifi_model.fit(sifi, n_sim=2000)

for w in caught:
    print("\n[warning]", w.message)

Follow the advice — usually with a value from a unimodal control block, or from
the literature — and the parameter stops floating:

In [ ]:
guided = btb.Model.from_data(sifi)
guided.set_prior(mu=2.0, sigma=3.0)
guided.fix("sigma_auditory", 0.9)          # measured elsewhere

guided_fit = guided.fit(sifi, n_sim=2000)
print("free parameters:", guided_fit.n_free)
print("p_common       :", round(guided_fit["p_common"], 3))

---
## 9. Many participants

Give the data a subject column and add `by="subject"`. Each participant is
fitted independently, and the group is summarised afterwards.

In [ ]:
rng = np.random.default_rng(0)
blocks = []
for index in range(6):
    truth = {"p_common": float(rng.uniform(0.35, 0.85)),
             "sigma_visual": float(rng.uniform(0.3, 0.7)),
             "sigma_auditory": float(rng.uniform(0.7, 1.3)),
             "mu_prior": 2.0, "sigma_prior": 3.0}
    blocks.append(generator.simulate(conditions, values=truth, n=40,
                                     seed=100 + index).to_data())

stimulus = np.concatenate([b.stimulus for b in blocks])
response = np.concatenate([b.response for b in blocks])
subject = np.concatenate([["s%02d" % (i + 1)] * len(b) for i, b in enumerate(blocks)])

group_data = btb.Data(stimulus, response, blocks[0].modalities, blocks[0].dimensions,
                      subject=subject, name="six subjects",
                      drop_absent_responses=False)
print(group_data)

In [ ]:
group_model = btb.Model.from_data(group_data)
group_model.set_prior(mu=2.0, sigma=3.0)

group = group_model.fit(group_data, by="subject", n_sim=1500)
print(group.summary())

In [ ]:
group.plot.params();

In [ ]:
group.to_frame().round(3)

---
## 10. Parameter recovery

Can this design recover the parameters at all? Simulate from known values, fit,
and compare. Four lines per replication.

In [ ]:
pairs = []
for index in range(8):
    truth = {"p_common": float(rng.uniform(0.2, 0.9)),
             "sigma_visual": float(rng.uniform(0.3, 0.8)),
             "sigma_auditory": float(rng.uniform(0.6, 1.4))}
    generated = generator.simulate(conditions, values=dict(truth, mu_prior=2.0,
                                                           sigma_prior=3.0),
                                   n=50, seed=200 + index).to_data()
    candidate = btb.Model.from_data(generated)
    candidate.set_prior(mu=2.0, sigma=3.0)
    pairs.append((truth, candidate.fit(generated, n_sim=1500)))

btb.plot.recovery(pairs);

The same loop answers *"how many trials do I need?"* — repeat it with `n=25`,
`50`, `100` and watch the scatter tighten around the diagonal.

---
## 11. Diagnostics: is my fit trustworthy?

Three habits catch almost every problem.

**(a) Use several starting points.** If they disagree, the objective has local
minima.

In [ ]:
checked = model.fit(data, n_sim=1500, n_start=3)
print("consistent across starts:", checked.optimization.converged_consistently)
for start in checked.optimization.starts:
    print("   start -> error %.3f" % start["fun"])

`converged_consistently` compares the objective values reached from every
start. If it is `False`, look at how large the spread actually is: a difference
of a fraction of a percent is usually just simulation noise, cured by raising
`n_sim`; a difference of several percent means genuine local minima, and calls
for more starts, tighter bounds, or a global optimiser
(`optimizer="differential_evolution"`).

**(b) Look at the objective landscape.** A profile shows whether the optimum is
a real minimum; a heat map shows parameters trading off against each other.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
btb.plot.landscape(fit.model, data, "p_common", values=fit.values,
                   n_grid=9, n_sim=600, ax=axes[0])
btb.plot.landscape(fit.model, data, "sigma_visual", "sigma_auditory",
                   values=fit.values, n_grid=7, n_sim=400, ax=axes[1])
fig.tight_layout()

**(c) Ask the fit itself.** `fit.diagnostics()` runs all three checks at once -
estimates on their bounds, parameters that different starting points disagree
about, and local minima - and says what to do about each.

In [ ]:
print(checked.diagnostics()["text"])

`btb.plot.params` shows the same thing visually: it draws the optimisation
bounds behind each estimate, so a dot sitting at the edge is immediately
obvious.

> **Prior parameters are often weakly identified.** In designs where the prior
> barely influences the responses, `mu_prior` and `sigma_prior` trade off
> against each other — which is why this notebook fixes them with
> `set_prior(mu=2.0, sigma=3.0)`. The classic GUI does the same by fixing
> `sigma_prior` at a very large value, i.e. a flat prior.

---
## 12. Posterior predictive checks

A good likelihood is not proof that the model reproduces the data. A posterior
predictive check simulates replicate data sets **of the same size** as the real
one and asks whether the observed summary statistics fall where the model says
they should.

In [ ]:
check = btb.posterior_predictive(fit, statistic="mean", n_replicates=300)
print(check.summary())

In [ ]:
check.plot();

Read `mean |z|`, not just the coverage: coverage only counts cells in or out of
an interval, while `|z|` measures *how far* off they are.

Note that these are **real data**, so even the best available model leaves some
misfit - a `max |z|` of several standard deviations on one cell is normal here,
and is itself informative about where the model falls short. What matters is the
*comparison*: a deliberately wrong model should be measurably worse.

In [ ]:
wrong = btb.posterior_predictive(null_fit, n_replicates=300)
print("causal inference : coverage %.0f%%, mean |z| %.2f, BIC %.0f"
      % (100*check.coverage, check.mean_abs_z, fit.bic))
print("no integration   : coverage %.0f%%, mean |z| %.2f, BIC %.0f"
      % (100*wrong.coverage, wrong.mean_abs_z, null_fit.bic))
print()
print("worst-missed cells for the wrong model:")
print(wrong.frame.reindex(wrong.frame["z"].abs().sort_values(ascending=False).index)
           .head(4)[["condition", "modality", "observed", "predicted_mean", "z"]]
           .round(3).to_string(index=False))

On simulated data, where the true model *is* available, the separation is much
sharper: the generating model gives `mean |z|` around 0.7 while a
no-integration model gives 1.6, and every badly-missed cell belongs to the
modality that should have been pulled by the other one.

---
## 13. Two stimulus dimensions

Some designs vary the stimuli along two dimensions at once — for example
numerosity **and** time (stimulus onset asynchrony). Each dimension gets its own
sensory noise and its own prior, and the evidence for a common cause multiplies
across them.

Parameter names simply gain the dimension suffix.

In [ ]:
two_d = btb.Model(["visual", "auditory"], ["numerosity", "time"])
two_d.set_response("discrete", levels=[0, 1, 2, 3, 4], dimension="numerosity")
for name, value in dict(p_common=0.6,
                        sigma_visual_numerosity=0.4, sigma_auditory_numerosity=0.8,
                        sigma_visual_time=60.0, sigma_auditory_time=40.0,
                        mu_prior_numerosity=2.0, sigma_prior_numerosity=3.0,
                        mu_prior_time=0.0, sigma_prior_time=400.0).items():
    two_d.fix(name, value)

grid = []
for v in (1., 2.):
    for a in (1., 2.):
        for soa in (0., 150., 300.):
            grid.append([[v, 0.0], [a, soa]])
grid = np.array(grid)

sim_2d = two_d.simulate(grid, n=2000, seed=3)
sim_2d.summary()[["stim_visual_numerosity", "stim_auditory_numerosity",
                  "stim_auditory_time", "mean_visual_numerosity",
                  "mean_auditory_numerosity", "p_common"]].round(3)

Look at the `p_common` column: at the same numerosity disparity, a larger
temporal offset lowers the probability of a common cause. Timing information
enters the same inference — nothing had to be special-cased.

Fitting works exactly as before; the time dimension has no responses, so it
informs the causal inference but not the likelihood.

In [ ]:
data_2d = sim_2d.to_data("two dimensional")
model_2d = btb.Model.from_data(data_2d)
model_2d.set_prior(mu=2.0, sigma=3.0, dimension="numerosity")
model_2d.set_prior(mu=0.0, sigma=400.0, dimension="time")

fit_2d = model_2d.fit(data_2d, n_sim=1500)
print("free parameters:", fit_2d.n_free)
print(dict((k, round(v, 3)) for k, v in fit_2d.free_values.items()))

Two dimensions have their own figures. The **integration window** shows
`p(C = 1)` as a function of the disparity on each dimension at once, and the
**joint plot** shows where the responses actually land in the plane spanned by
the two dimensions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
btb.plot.posterior_common_2d(two_d, ax=axes[0])
axes[0].set_title("integration window")

both = btb.Model(["visual", "auditory"], ["space", "time"], response="continuous")
for name, value in dict(p_common=0.6, sigma_visual_space=2.0, sigma_auditory_space=9.0,
                        sigma_visual_time=30.0, sigma_auditory_time=80.0,
                        mu_prior_space=0.0, sigma_prior_space=15.0,
                        mu_prior_time=0.0, sigma_prior_time=300.0).items():
    both.fix(name, value)
btb.plot.posterior_common_2d(both, ax=axes[1])
axes[1].set_title("space x time")
fig.tight_layout()

In [ ]:
space_time = np.array([[[v, 0.0], [a, t]] for v in (-10.0, 10.0)
                       for a in (-10.0, 10.0) for t in (-200.0, 0.0, 200.0)])
btb.plot.joint(both.simulate(space_time, n=1500, seed=5), max_conditions=6);

Read the joint plot condition by condition. Where both disparities are zero the
two clouds collapse onto each other; where both are large each response sits on
its own stimulus; where only one is large the noisy modality stretches into the
characteristic partially-integrated shape.

Compare the fitted values with the ones used to generate the data. The
numerosity parameters come back closely; the **time** parameters do not, because
no temporal response was ever recorded — they are constrained only indirectly,
through their effect on the causal inference. In a real analysis you would
normally fix them at values measured in a temporal-order or
simultaneity-judgement task:

```python
model_2d.fix("sigma_visual_time", 60.0)
model_2d.fix("sigma_auditory_time", 40.0)
```

---
## 14. Judging the causal structure itself

Some experiments do not ask *where* or *how many* - they ask **whether the
signals belong together**. "Did they come from the same place?", "Did the rubber
hand feel like your own?", "Were they simultaneous?" The report *is* the causal
inference, so the response model reads the posterior directly.

Such a dimension has a response but **no stimulus**, and the toolbox recognises
it on that basis.

In [ ]:
rhi = btb.Model(["visual", "tactile"], ["time", "ownership"], name="rubber hand")
rhi.set_response("unity", dimension="ownership")
for name, value in dict(p_common=0.55, sigma_visual_time=120.0,
                        sigma_tactile_time=160.0, mu_prior_time=0.0,
                        sigma_prior_time=800.0).items():
    rhi.fix(name, value)

asynchronies = [-500., -300., -150., 0., 150., 300., 500.]
trials = np.array([[[0.0, np.nan], [a, np.nan]] for a in asynchronies])
rhi_sim = rhi.simulate(trials, n=3000, seed=4)

illusion = np.nanmean(rhi_sim.responses[:, :, 0, 1], axis=1)
plt.figure(figsize=(4.6, 3.4))
plt.plot(asynchronies, illusion, "o-")
plt.xlabel("visuotactile asynchrony (ms)"); plt.ylabel("P(illusion)")
plt.ylim(0, 1); plt.title("the model's ownership curve");

The inverted U falls out of the causal inference: at large asynchrony the
evidence for a common cause collapses, and with it the illusion.

Fitting such data works like anything else - and the diagnostics earn their keep
here.

In [ ]:
rhi_data = rhi_sim.to_data("rhi")
print("reported dimensions:", rhi_data.reported_dimensions)

candidate = btb.Model.from_data(rhi_data)
print("auto-detected response models:",
      [None if r is None else r.kind for r in candidate.response_models])
candidate.set_prior(mu=0.0, sigma=800.0, dimension="time")

rhi_fit = candidate.fit(rhi_data, n_sim=1500, n_start=4)
print()
print(rhi_fit.diagnostics()["text"])

The design cannot separate the two sensory noises: a **symmetric** yes/no
judgement constrains essentially their sum. Tie them, and the fit becomes
well-posed:

In [ ]:
tied = btb.Model.from_data(rhi_data)
tied.set_prior(mu=0.0, sigma=800.0, dimension="time")
tied.tie("sigma_tactile_time", "sigma_visual_time")

tied_fit = tied.fit(rhi_data, n_sim=1500, n_start=4)
print(tied_fit.diagnostics()["text"])
print()
print("tied sigma      : %.1f" % tied_fit["sigma_visual_time"])
print("true combination: %.1f" % np.sqrt((120.0**2 + 160.0**2) / 2))

A unity judgement can also be collected **alongside** a continuous response, as
in audiovisual localisation experiments that ask "where was it?" and "was it one
event or two?" on every trial. Both responses enter the same likelihood.

In [ ]:
joint_model = btb.Model(["visual", "auditory"], ["space", "unity"])
joint_model.set_response("continuous", dimension="space")
joint_model.set_response("unity", dimension="unity")
for name, value in dict(p_common=0.6, sigma_visual_space=2.0,
                        sigma_auditory_space=9.0, mu_prior_space=0.0,
                        sigma_prior_space=15.0).items():
    joint_model.fix(name, value)

joint_grid = np.array([[[v, np.nan], [a, np.nan]]
                       for v in (-12., -4., 4., 12.) for a in (-12., -4., 4., 12.)])
joint_data = joint_model.simulate(joint_grid, n=60, seed=11).to_data("joint")

joint_fit = btb.Model.from_data(joint_data).fit(joint_data, n_sim=2000)
print("dimensions fitted:", joint_data.reported_dimensions)
print("sigma_visual_space  %.2f (true 2.00)" % joint_fit["sigma_visual_space"])
print("sigma_auditory_space %.2f (true 9.00)" % joint_fit["sigma_auditory_space"])
print("p_common             %.2f (true 0.60)" % joint_fit["p_common"])

---
## 15. More than two modalities

The causal inference has a closed form for any number of signals, so a
trimodal model needs no new code.

In [ ]:
trimodal = btb.Model(["visual", "auditory", "tactile"], ["space"],
                     response="continuous")
for name, value in dict(sigma_visual=1.0, sigma_auditory=1.0, sigma_tactile=1.0,
                        p_common=0.5, mu_prior=0.0, sigma_prior=10.0).items():
    trimodal.fix(name, value)

for disparity in (0.0, 1.0, 3.0, 10.0):
    condition = np.array([[[-disparity], [disparity], [0.0]]])
    result = trimodal.simulate(condition, n=2000)
    print("disparity %5.1f  ->  p(C=1) = %.3f   responses = %s"
          % (disparity, result.p_common.mean(),
             np.round(result.responses[0].mean(axis=0).ravel(), 2)))

From complete fusion (all three responses near 0) to complete segregation (each
response at its own stimulus), through the causal inference alone.

---
## 16. Continuous responses: spatial localisation

Everything so far used discrete counts. Continuous reports — a pointed or
matched location — work identically; only the response model changes.

In [ ]:
localisation = btb.Model(["visual", "auditory"], ["space"], response="continuous")
truth = {"p_common": 0.6, "sigma_visual": 2.0, "sigma_auditory": 9.0,
         "mu_prior": 0.0, "sigma_prior": 15.0}
space_conditions = [(v, a) for v in (-12., -4., 4., 12.) for a in (-12., -4., 4., 12.)]

space_data = localisation.simulate(space_conditions, values=truth,
                                   n=60, seed=101).to_data("localisation")
space_fit = btb.Model.from_data(space_data).fit(space_data, n_sim=2500)

for name, value in truth.items():
    print("  %-16s true %7.3f   fitted %7.3f" % (name, value, space_fit.values[name]))

`p_common` and both sensory noises come back accurately. `mu_prior` does not,
and that is a property of this design rather than a failure of the fit: with
`sigma_prior = 15` the prior contributes under 2% of the weight of the visual
estimate, so the data barely constrain where its mean sits. Part 11 shows how to
detect this, and `set_prior` is how to fix it.

In [ ]:
btb.plot.fit(space_fit, max_conditions=16);

Two views of the same fit:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
btb.plot.bias(space_fit, ax=axes[0])
btb.plot.posterior_common(space_fit.model, values=space_fit.values, ax=axes[1])
fig.tight_layout()

In [ ]:
btb.plot.bias(space_fit, relative=False);

> **A note on continuous likelihoods.** The likelihood of a continuous response
> is a kernel density estimate of the simulated responses. The bandwidth is
> derived from the spread of the responses **within** a condition, separately per
> modality — a bandwidth taken from the pooled spread across conditions is far
> too wide and biases the estimated sensory noise downwards. See
> `docs/responses.md`.

---
## 17. Objectives and optimisers

**Objectives** (`objective=`):

| Name | Meaning | AIC/BIC? |
| --- | --- | --- |
| `"mll"` | minus log likelihood (default) | yes |
| `"sse"` | squared error between distributions | no |
| `"r2"` | minus squared correlation | no |
| `"emd"` | Wasserstein distance between response samples | no |

**Optimisers** (`optimizer=`):

| Name | When |
| --- | --- |
| `"powell"` | default; robust to the mild noise of a simulated objective |
| `"neldermead"` | alternative simplex search |
| `"differential_evolution"` | global search; slow but insensitive to the start |
| `"vbmc"` | approximate **posterior** over parameters (needs `pyvbmc`) |

A global search is the best cross-check when a fit looks suspicious:

In [ ]:
# Objective values are only comparable at the same n_sim, so refit Powell here.
local_fit = model.fit(data, optimizer="powell", n_sim=1200)
global_fit = model.fit(data, optimizer="differential_evolution", n_sim=1200,
                       options={"maxiter": 15, "popsize": 8})

print("Powell                : error %.3f" % local_fit.error)
print("differential evolution: error %.3f" % global_fit.error)
print("parameters agree      :",
      {k: (round(local_fit[k], 3), round(global_fit[k], 3))
       for k in local_fit.free_values})

With `optimizer="vbmc"` the fit also carries an approximate posterior:

```python
posterior_fit = model.fit(data, optimizer="vbmc", n_sim=1000)
samples = posterior_fit.optimization.extras["posterior_samples"]
posterior_fit.optimization.extras["elbo"]
```

VBMC is expensive — budget roughly a hundred times a Powell fit.

---
## 18. Saving and reproducibility

A fit serialises to JSON, and the model specification round-trips, so an
analysis can be reconstructed exactly.

In [ ]:
import tempfile, os

path = os.path.join(tempfile.gettempdir(), "my_fit.json")
fit.save(path)

record = btb.load_fit(path)
restored = btb.Model.from_dict(record["model"])
print("restored free parameters:", restored.params.estimated_names)
print("stored BIC              :", round(record["bic"], 2))

group.save_csv(os.path.join(tempfile.gettempdir(), "group_parameters.csv"))
sim.save_csv(os.path.join(tempfile.gettempdir(), "simulated_trials.csv"))
print("exports written")

Fits are reproducible by construction: the simulation noise is drawn from a
fixed seed and reused across optimiser iterations (*common random numbers*), so
running the same fit twice gives the same answer.

In [ ]:
first = model.fit(data, n_sim=1000)
second = model.fit(data, n_sim=1000)
print("identical:", first.error == second.error)

---
## Where to go next

The full API reference ships with the package, in `bcitoolbox/docs/`:

| Page | Contents |
| --- | --- |
| `index.md` | overview and design principles |
| `quickstart.md` | five short worked examples |
| `data.md` | loading any file layout |
| `model.md` | building, fitting, simulating |
| `parameters.md` | free / fixed / bounded / tied |
| `responses.md` | task-specific observation models |
| `results.md` | `Fit`, `FitGroup`, `Simulation`, `compare` |
| `plotting.md` | every figure |
| `recovery.md` | `recover`, posterior predictive checks, diagnostics |
| `engine.md`, `likelihood.md`, `optimizers.md` | the numerical core |
| `theory.md` | the equations, as implemented, with validation results |

```python
import os, bcitoolbox
print(os.path.join(os.path.dirname(bcitoolbox.__file__), "docs"))
```

The graphical interface is still one call away — `btb.gui()` and `btb.gui2d()` —
and is documented at <https://bci-toolbox.readthedocs.io>.

**Citing:** Zhu, H., Beierholm, U., & Shams, L. (2024). BCI Toolbox: An
open-source Python package for the Bayesian causal inference model.
*PLOS Computational Biology*, 20(7), e1011791.